# Lab 6 — *Sound of Muzak*: one rhythmic object through many representations

This is the flagship case study. It uses a **synthetic reconstruction** of the drum pattern found in Bruno Di Giorgi's teaching/research material, not the commercial recording.

For listening comparison, use the external reference supplied for the course: https://youtu.be/cAtsmJMw0kU?si=ylsjelUjdi_f4_Ou

The point of the lab is to follow the *same rhythmic object* from symbolic structure to audio and back to signal-derived representations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d
from IPython.display import Audio, display

np.set_printoptions(precision=3, suppress=True)

def unbiased_autocorrelation(x, max_lag=None, normalize=True):
    x = np.asarray(x, dtype=float).reshape(-1)
    if max_lag is None:
        max_lag = len(x)-1
    max_lag = min(int(max_lag), len(x)-1)
    r = np.empty(max_lag+1)
    for tau in range(max_lag+1):
        a=x[:len(x)-tau]; b=x[tau:]
        r[tau] = np.dot(a,b)/max(1,len(a))
    if normalize and r[0] > 0:
        r = r/r[0]
    return r

def phase_autocorrelation(x, lags, unbiased=True):
    x=np.asarray(x,dtype=float).reshape(-1)
    lags=np.asarray(lags,dtype=int)
    P=np.zeros((len(lags), int(lags.max())))
    C=np.zeros_like(P)
    N=len(x)
    for ir,tau in enumerate(lags):
        for phi in range(tau):
            idx=np.arange(phi, N-tau, tau, dtype=int)
            if len(idx):
                P[ir,phi]=np.sum(x[idx]*x[idx+tau])
                C[ir,phi]=len(idx)
    if unbiased:
        return P/np.where(C==0,1,C), C
    return P, C

def frame_signal(x, frame_length, hop_length, center=True):
    x=np.asarray(x,float).reshape(-1)
    if center:
        p=frame_length//2
        x=np.pad(x,(p,p))
    starts=np.arange(0,max(1,len(x)-frame_length+1),hop_length,dtype=int)
    return np.stack([x[s:s+frame_length] for s in starts],axis=1), starts

def rhythmogram(x, frame_length=256, hop_length=32, lag_min=8, lag_max=96):
    frames, starts = frame_signal(x, frame_length, hop_length, center=True)
    lags=np.arange(lag_min,lag_max+1)
    R=np.zeros((len(lags),frames.shape[1]))
    w=np.hanning(frame_length)
    for j in range(frames.shape[1]):
        f=(frames[:,j]-frames[:,j].mean())*w
        ac=unbiased_autocorrelation(f, lag_max, normalize=False)
        R[:,j]=np.maximum(ac[lags],0)
        m=R[:,j].max()
        if m>0: R[:,j]/=m
    return R,lags,starts

def dp_tempo_path(R, lags, smoothness=30.0):
    R=np.asarray(R,float)
    eps=1e-10
    obs=-np.log(R+eps)
    nP,nT=R.shape
    cost=np.full((nP,nT),np.inf); back=np.zeros((nP,nT),dtype=int)
    cost[:,0]=obs[:,0]
    lp=np.log(np.asarray(lags,float))
    trans=smoothness*(lp[:,None]-lp[None,:])**2
    for t in range(1,nT):
        M=cost[:,t-1][None,:]+trans
        back[:,t]=np.argmin(M,axis=1)
        cost[:,t]=obs[:,t]+M[np.arange(nP),back[:,t]]
    path=np.zeros(nT,dtype=int); path[-1]=np.argmin(cost[:,-1])
    for t in range(nT-2,-1,-1): path[t]=back[path[t+1],t+1]
    return np.asarray(lags)[path]

def simple_novelty(x, fs, n_fft=1024, hop=256):
    f,t,Z=signal.stft(x,fs=fs,window='hann',nperseg=n_fft,noverlap=n_fft-hop,boundary=None,padded=False)
    mag=np.abs(Z)
    flux=np.maximum(mag[:,1:]-mag[:,:-1],0).sum(axis=0)
    flux=np.r_[0,flux]
    if flux.max()>0: flux/=flux.max()
    return flux,t

PATTERN = {
    'Splash': 'S[16] 6---------------------------',
    'Closed hi-hat': 'S[16] --5-9-5-9-5-9-5-9-5-9-5-----',
    'Open hi-hat': 'S[16] ------------------------9---',
    'Snare': 'S[16] -----9------9------9------9-',
    'Kick': 'S[16] 9--9---9--9---9--9---9--9---',
}
def symbols(line): return line.split(']',1)[1].strip().replace(' ','')

def _burst(fs,dur,f0,noise=0.0,decay=.03,seed=0):
    n=np.arange(int(dur*fs)); env=np.exp(-n/(decay*fs))
    rng=np.random.default_rng(seed)
    y=np.sin(2*np.pi*f0*n/fs)
    if noise: y=(1-noise)*y+noise*rng.standard_normal(len(n))
    return env*y

def synthesize_pattern(pattern,bpm=90,fs=22050,repeats=4):
    step=60/bpm/4
    L=max(len(symbols(v)) for v in pattern.values())
    dur=(L*repeats+4)*step
    y=np.zeros(int(dur*fs))
    voices={
      'Kick':_burst(fs,.22,62,.05,.055,1),
      'Snare':_burst(fs,.16,190,.75,.035,2),
      'Closed hi-hat':_burst(fs,.07,5000,.95,.018,3),
      'Open hi-hat':_burst(fs,.24,4200,.95,.075,4),
      'Splash':_burst(fs,.45,2800,.95,.12,5),
    }
    for rep in range(repeats):
      for name,line in pattern.items():
        s=symbols(line)
        for i,ch in enumerate(s):
          if ch.isdigit():
            amp=int(ch)/9
            p=int(round((rep*L+i)*step*fs)); v=voices[name]
            q=min(len(y),p+len(v)); y[p:q]+=amp*v[:q-p]
    y/=max(1,np.max(np.abs(y)))
    return y,fs,step,L

## 1. Inspect the symbolic components

In [ ]:
names=list(PATTERN)
L=max(len(symbols(v)) for v in PATTERN.values())
M=np.zeros((len(names),L))
for r,name in enumerate(names):
    for i,ch in enumerate(symbols(PATTERN[name])):
        if ch.isdigit(): M[r,i]=int(ch)/9
plt.figure(figsize=(13,3.5)); plt.imshow(M,aspect='auto',interpolation='nearest'); plt.yticks(range(len(names)),names); plt.xlabel('sixteenth-note grid cell'); plt.title('Symbolic drum layers'); plt.show()

Before separating the rows analytically, listen for the **compound gesture**. The course interpretation is that expert performance need not be understood as five independent clocks: components can be perceptually and motorically grouped into a higher-level rhythmic cluster.

In [ ]:
x,fs,step,L=synthesize_pattern(PATTERN,bpm=90,repeats=5)
display(Audio(x,rate=fs))
plt.figure(figsize=(12,2.5)); plt.plot(np.arange(len(x))/fs,x,lw=.5); plt.xlim(0,min(10,len(x)/fs)); plt.xlabel('time [s]'); plt.title('Synthetic performance'); plt.show()

## 2. Audio → onset evidence

In [ ]:
odf,todf=simple_novelty(x,fs,n_fft=1024,hop=256)
plt.figure(figsize=(12,3)); plt.plot(todf,odf); plt.xlabel('time [s]'); plt.ylabel('novelty'); plt.title('Spectral-flux onset evidence'); plt.show()

## 3. Global autocorrelation and APM

In [ ]:
maxlag=min(220,len(odf)-1)
r=unbiased_autocorrelation(odf,maxlag)
plt.figure(figsize=(11,3)); plt.plot(r); plt.xlabel('lag [ODF samples]'); plt.title('Global autocorrelation'); plt.show()
lags=np.arange(8,min(90,maxlag+1))
P,_=phase_autocorrelation(odf,lags,unbiased=True)
plt.figure(figsize=(10,6)); plt.imshow(P,origin='lower',aspect='auto',extent=[0,P.shape[1]-1,lags[0],lags[-1]]); plt.xlabel('phase φ'); plt.ylabel('lag τ'); plt.title('Phase autocorrelation'); plt.colorbar(); plt.show()

## 4. Time-varying periodicity and a tempo path

In [ ]:
R,lagsR,starts=rhythmogram(odf,frame_length=min(256,max(64,len(odf)//2)),hop_length=12,lag_min=8,lag_max=min(80,max(12,len(odf)//3)))
path=dp_tempo_path(R,lagsR,smoothness=25)
plt.figure(figsize=(12,5)); plt.imshow(R,origin='lower',aspect='auto',extent=[0,R.shape[1]-1,lagsR[0],lagsR[-1]]); plt.plot(np.arange(len(path)),path,'w-',lw=2,label='DP period path'); plt.xlabel('frame'); plt.ylabel('period'); plt.legend(); plt.title('Rhythmogram and tracked period'); plt.show()

## 5. Return to the algebra

Earlier in the chapter, *Sound of Muzak* is used to discuss a systemic \(7{:}4\) organization and a translated component (coset) on a common temporal lattice. The signal-analysis representations do not recover that algebraic description by magic; rather, they expose periodic and phase regularities from which structural hypotheses can be formed.

**Question:** which aspects of the symbolic cluster are visible in autocorrelation, which require phase, and which depend on higher-level musical interpretation?

### Attribution

The synthetic pattern and original phase/rhythm-analysis experiments derive from material developed by **Bruno Di Giorgi** during his PhD at Politecnico di Milano. This notebook is a modern CMRM teaching reconstruction. The Porcupine Tree recording is not redistributed.